# HadGEM PPE and CMIP6 precipitation distributions

This notebook compares the GA7, GA8, and GA9 perturbed-parameter ensembles with the CMIP6 model distribution. It focuses on quantities relevant to the manuscript and reviewer response:

- global-mean historical precipitation;
- global-mean precipitation response per degree of warming (dPdK);
- spatial amplitude of dPdK patterns;
- leave-one-out distance from the other patterns in the same archive.

The CMIP6 climatology archive contains 47 models. The dPdK comparison uses only models with a precomputed precipitation response and matched global-temperature change. Multiple realizations are averaged within each CMIP6 model before comparing distributions, so models with many realizations do not receive extra weight. Missing temperature-response data therefore reduce the dPdK sample but do not remove a model from the climatology comparison.


## Imports and data locations

The paths are kept together here so the notebook is easy to move or update. CMIP6 dPdK files are loaded individually because they are not included in the older 47-model aggregate file.


In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

researchpath = Path("/Users/evanwellmeyer/Documents/research")
hadgempath = researchpath / "HadGEM"
cmip6path = researchpath / "CMIP6"

hadgemprpath = hadgempath / "GA789_PR_his_rg128.nc"
hadgemresponsepath = hadgempath / "GA789_dPdK_rg128.nc"
cmip6prpath = cmip6path / "CMIP6_PR_his_rg128.nc"
cmip6responsepath = cmip6path / "dPdK"

secondsperyear = 365.25 * 24 * 60 * 60
familyorder = ["GA7", "GA8", "GA9"]
familycolors = {"GA7": "#4477aa", "GA8": "#ee6677", "GA9": "#228833"}
cmip6color = "#aa3377"


## Helper functions

All spatial summaries use cosine-latitude weighting. The leave-one-out RMSE compares each pattern with the mean of the remaining patterns in its group.


In [ ]:
def spatialnames(field):
    latname = "latitude" if "latitude" in field.dims else "lat"
    lonname = "longitude" if "longitude" in field.dims else "lon"
    return latname, lonname


def weightedmean(field):
    latname, lonname = spatialnames(field)
    weights = np.cos(np.deg2rad(field[latname]))
    return field.weighted(weights).mean((latname, lonname), skipna=True)


def weightedrmse(field):
    return np.sqrt(weightedmean(field ** 2))


def patternamplitude(field):
    return weightedrmse(field - weightedmean(field))


def modelname(path):
    name = path.name.split("_Amon_", 1)[1]
    return name.split("_ssp585_", 1)[0]


def standardizegrid(field):
    rename = {}
    if "lat" in field.dims:
        rename["lat"] = "latitude"
    if "lon" in field.dims:
        rename["lon"] = "longitude"
    return field.rename(rename)


def leaveoneoutrmse(field):
    count = field.sizes["realization"]
    total = field.sum("realization", skipna=False)
    reference = (total - field) / (count - 1)
    return weightedrmse(field - reference)


## Load the HadGEM ensembles

The family label comes from the realization name. All paired historical and p4K members in the supplied GA7/GA8/GA9 files are retained.


In [ ]:
hadgempr = xr.open_dataset(hadgemprpath)["PR"]
hadgemresponse = xr.open_dataset(hadgemresponsepath)["dPdK"]

hadgempr, hadgemresponse = xr.align(hadgempr, hadgemresponse, join="inner")
hadgemfamily = xr.DataArray(
    [str(name).split("_", 1)[0] for name in hadgempr.realization.values],
    coords={"realization": hadgempr.realization},
    dims="realization",
    name="family",
)

print(f"HadGEM paired members: {hadgempr.sizes['realization']}")
for family in familyorder:
    print(f"  {family}: {int((hadgemfamily == family).sum())}")


## Load CMIP6 climatology and dPdK

Each available CMIP6 dPdK field is converted from kg m$^{-2}$ s$^{-1}$ K$^{-1}$ to mm yr$^{-1}$ K$^{-1}$, interpolated to the HadGEM grid, and then averaged across realizations belonging to the same model. The code explicitly intersects these model names with the climatology archive; a model lacking dPdK remains available for the historical-precipitation plot.


In [ ]:
cmip6pr = xr.open_dataset(cmip6prpath)["PR"]
targetlatitude = hadgemresponse.latitude
targetlongitude = hadgemresponse.longitude

responsefiles = sorted(cmip6responsepath.glob("dPdK_Amon_*.nc"))
responsesbymodel = defaultdict(list)

for path in responsefiles:
    dataset = xr.open_dataset(path)
    response = standardizegrid(dataset["dPdK"]).squeeze(drop=True)
    response = response * secondsperyear
    response = response.interp(
        latitude=targetlatitude,
        longitude=targetlongitude,
    ).load()
    responsesbymodel[modelname(path)].append(response)
    dataset.close()

cmip6models = [str(name) for name in cmip6pr.realization.values]
matchedmodels = sorted(set(cmip6models).intersection(responsesbymodel))
missingmodels = sorted(set(cmip6models).difference(responsesbymodel))

cmip6responses = []
for name in matchedmodels:
    response = xr.concat(
        responsesbymodel[name], dim="member", coords="minimal", compat="override"
    ).mean("member")
    cmip6responses.append(response.expand_dims(realization=[name]))

cmip6response = xr.concat(cmip6responses, dim="realization")

availability = pd.DataFrame({
    "archive": ["HadGEM GA7/GA8/GA9", "CMIP6"],
    "climatology_count": [hadgempr.sizes["realization"], len(cmip6models)],
    "dpdk_count": [hadgemresponse.sizes["realization"], len(matchedmodels)],
    "comparison_unit": ["member", "model mean"],
})
display(availability)

print(f"CMIP6 dPdK files: {len(responsefiles)}")
print(f"CMIP6 models with matched climatology and dPdK: {len(matchedmodels)}")
print(f"CMIP6 climatology models without dPdK: {len(missingmodels)}")
if missingmodels:
    print("Missing dPdK models:", ", ".join(missingmodels))


## Historical precipitation distribution

This comparison uses all 47 CMIP6 climatology models, whether or not temperature-response data are available. The HadGEM boxes show member-level PPE spread; the CMIP6 box shows model-to-model spread.


In [ ]:
hadgemprmean = weightedmean(hadgempr)
cmip6prmean = weightedmean(cmip6pr)

prgroups = []
prlabels = []
prcolors = []

for family in familyorder:
    values = hadgemprmean.where(hadgemfamily == family, drop=True).values
    prgroups.append(values)
    prlabels.append(family)
    prcolors.append(familycolors[family])

prgroups.append(cmip6prmean.values)
prlabels.append("CMIP6")
prcolors.append(cmip6color)

figure, axis = plt.subplots(figsize=(8, 5))
boxes = axis.boxplot(prgroups, tick_labels=prlabels, patch_artist=True, showfliers=False)
for box, color in zip(boxes["boxes"], prcolors):
    box.set_facecolor(color)
    box.set_alpha(0.65)
axis.set_ylabel("Global-mean precipitation (mm yr$^{-1}$)")
axis.set_title("Historical precipitation distributions")
axis.grid(axis="y", alpha=0.25)
plt.show()


## dPdK distribution

The CMIP6 response sample is limited to models with matched precipitation and temperature information. Because CMIP6 realizations are averaged within model first, this is a model-weighted comparison rather than a realization-weighted one.


In [ ]:
hadgemresponsemean = weightedmean(hadgemresponse)
cmip6responsemean = weightedmean(cmip6response)
hadgemamplitude = patternamplitude(hadgemresponse)
cmip6amplitude = patternamplitude(cmip6response)

figure, axes = plt.subplots(1, 2, figsize=(12, 5))

for axis, hadgemmetric, cmip6metric, ylabel, title in [
    (
        axes[0],
        hadgemresponsemean,
        cmip6responsemean,
        "Global-mean dPdK (mm yr$^{-1}$ K$^{-1}$)",
        "Mean precipitation response",
    ),
    (
        axes[1],
        hadgemamplitude,
        cmip6amplitude,
        "Spatial SD of dPdK (mm yr$^{-1}$ K$^{-1}$)",
        "Response-pattern amplitude",
    ),
]:
    groups = []
    labels = []
    colors = []
    for family in familyorder:
        groups.append(hadgemmetric.where(hadgemfamily == family, drop=True).values)
        labels.append(family)
        colors.append(familycolors[family])
    groups.append(cmip6metric.values)
    labels.append("CMIP6")
    colors.append(cmip6color)
    boxes = axis.boxplot(groups, tick_labels=labels, patch_artist=True, showfliers=False)
    for box, color in zip(boxes["boxes"], colors):
        box.set_facecolor(color)
        box.set_alpha(0.65)
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.25)

figure.tight_layout()
plt.show()


## Pattern disagreement within each archive

For each HadGEM member, the reference is the mean of the other members in the same GA family. For each CMIP6 model, the reference is the mean of the other CMIP6 models. This is descriptive archive disagreement, not predictive uncertainty.


In [ ]:
hadgemloo = xr.full_like(hadgemresponsemean, np.nan)
for family in familyorder:
    mask = hadgemfamily == family
    values = hadgemresponse.where(mask, drop=True)
    hadgemloo.loc[{"realization": values.realization}] = leaveoneoutrmse(values)

cmip6loo = leaveoneoutrmse(cmip6response)

groups = []
labels = []
colors = []
for family in familyorder:
    groups.append(hadgemloo.where(hadgemfamily == family, drop=True).values)
    labels.append(family)
    colors.append(familycolors[family])
groups.append(cmip6loo.values)
labels.append("CMIP6")
colors.append(cmip6color)

figure, axis = plt.subplots(figsize=(8, 5))
boxes = axis.boxplot(groups, tick_labels=labels, patch_artist=True, showfliers=False)
for box, color in zip(boxes["boxes"], colors):
    box.set_facecolor(color)
    box.set_alpha(0.65)
axis.set_ylabel("Leave-one-out pattern RMSE (mm yr$^{-1}$ K$^{-1}$)")
axis.set_title("Within-archive dPdK pattern disagreement")
axis.grid(axis="y", alpha=0.25)
plt.show()


## Numerical summary

The percentile range is the 5th to 95th percentile. These values provide a compact statistical comparison for deciding whether the HadGEM PPE distributions are unusual relative to CMIP6.


In [ ]:
def summaryrow(name, count, precipitation, response, amplitude, disagreement):
    return {
        "group": name,
        "count": count,
        "pr_median": float(np.nanmedian(precipitation)),
        "pr_p05": float(np.nanpercentile(precipitation, 5)),
        "pr_p95": float(np.nanpercentile(precipitation, 95)),
        "dpdk_median": float(np.nanmedian(response)),
        "dpdk_p05": float(np.nanpercentile(response, 5)),
        "dpdk_p95": float(np.nanpercentile(response, 95)),
        "amplitude_median": float(np.nanmedian(amplitude)),
        "loo_rmse_median": float(np.nanmedian(disagreement)),
    }


rows = []
for family in familyorder:
    mask = hadgemfamily == family
    rows.append(summaryrow(
        family,
        int(mask.sum()),
        hadgemprmean.where(mask, drop=True).values,
        hadgemresponsemean.where(mask, drop=True).values,
        hadgemamplitude.where(mask, drop=True).values,
        hadgemloo.where(mask, drop=True).values,
    ))

cmip6prmatched = cmip6pr.sel(realization=matchedmodels)
rows.append(summaryrow(
    "CMIP6 matched dPdK",
    len(matchedmodels),
    weightedmean(cmip6prmatched).values,
    cmip6responsemean.values,
    cmip6amplitude.values,
    cmip6loo.values,
))

summary = pd.DataFrame(rows).set_index("group")
display(summary.round(2))


## Interpretation cautions

- The HadGEM distributions describe perturbed parameters within one model framework; CMIP6 describes structural differences among models. Similar or different spread does not by itself identify the source of uncertainty.
- HadGEM p4K minus historical responses and CMIP6 late-SSP585 minus historical responses are not identical experimental designs. The comparison is a useful scale check, not a controlled attribution experiment.
- The dPdK plots omit CMIP6 models without matched temperature-response information. The availability table and missing-model list make this selection visible.
- A formal manuscript statistic should report the exact model list and likely use one value per CMIP6 model, as done here.
